In [1]:
#from skimage import feature
import pandas as pd
import numpy as np
#import matplotlib.pyplot as plt
from scipy import signal
import os
from pathlib import Path
import sys
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import accuracy_score, classification_report, average_precision_score
from sklearn.metrics import precision_recall_curve

from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier as DTC
from sklearn.ensemble import RandomForestClassifier as RFC
from sklearn.neighbors import KNeighborsClassifier as KNN
import xgboost as XGB


In [2]:
parent_folder = Path().resolve().parent
src_path = parent_folder / 'src'
sys.path.append(str(src_path))

from tools import get_embedding_birdnet

#env to use: clef

In [4]:
root_folder='../data/train_data/embedding/holderried/'
df_pos1 = get_embedding_birdnet(root_folder, 1)
df_pos1["filename"] = df_pos1["embed_name"].str.split(".").str[0]
df_pos1["selec"] = df_pos1["filename"].astype(int)

In [ ]:
df_train = pd.read_csv("Holderried_pos_train_1956.csv") # Holder selections for training

In [ ]:
df_pos1 = df_pos1.merge(
    df_train[["selec", "sound.files", "recording.id"]],
    on="selec",
    how="left"
)

df_pos1 = df_pos1.dropna() # this removes extra "selec" columns

In [7]:
len(df_pos1)

1897

In [6]:
df_pos1.head(1)

,embed_name,embedding,filename,selec,sound.files,recording.id
1,1000.birdnet.embeddings.txt,"[1.4150934, 1.4794579, 0.37235585, 0.4117538, ...",1000,1000,20200625_194506.WAV,15562.0


In [ ]:
root_folder='../data/XC_long/' # XC Long positive records
df_pos2 = get_embedding_birdnet(root_folder+"embedding/", 1)
df_pos2["filename"] = df_pos2["embed_name"].str.split("-").str[0].str[:-1]
df_pos2["wavname"] = df_pos2["embed_name"].str.split(".").str[0] + ".wav"
preds = pd.read_csv(root_folder + 'birdnet_probs/1/predictions.csv')
preds["wavname"] = preds["filename"]
df_pos2 = df_pos2.merge(preds,on="wavname",how="left")

In [29]:
df_pos2.head(2)

,embed_name,embedding,filename_x,wavname,filename_y,confidence
0,XC1009958 - Eurasian Woodcock - Scolopax rusti...,"[0.0, 0.2501804, 0.3810754, 0.15622263, 0.0083...",XC1009958,XC1009958 - Eurasian Woodcock - Scolopax rusti...,XC1009958 - Eurasian Woodcock - Scolopax rusti...,0.000000
1,XC1009958 - Eurasian Woodcock - Scolopax rusti...,"[0.019925147, 0.17384617, 0.74116606, 0.0, 0.9...",XC1009958,XC1009958 - Eurasian Woodcock - Scolopax rusti...,XC1009958 - Eurasian Woodcock - Scolopax rusti...,0.740717


In [10]:
root_folder='../data/train_data/embedding/xc/' # XC Inintial Submission positive records
df_pos3 = get_embedding_birdnet(root_folder, 1)
df_pos3["filename"] = df_pos3["embed_name"].str.split("-").str[0].str[:-1]

In [11]:
df_pos = pd.concat([df_pos1, df_pos2, df_pos3], ignore_index=True)

In [12]:
len(df_pos)

3016

In [34]:
root_folder='../data/FVA_non_Eurasian_segs/embedding/seed2/'
df_neg1 = get_embedding_birdnet(root_folder, 0)
df_neg1["filename"] = df_neg1["embed_name"].str.split("_").str[0]

In [ ]:
root_folder='../data/XC_long/embedding/' # XC-Long positive records
df_neg2 = get_embedding_birdnet(root_folder, 0)
df_neg2["filename"] = df_neg2["embed_name"].str.split("-").str[0].str[:-1]

In [ ]:
root_folder='../data/train_data/embedding/northwest_DE/' # North DE records
df_neg3 = get_embedding_birdnet(root_folder, 0)
df_neg3["filename"] = df_neg3["embed_name"].str.split("_").str[0]

In [16]:
root_folder='../data/audiomoth_DE/embedding/' # Audiomoth DE negative records
df_neg4 = get_embedding_birdnet(root_folder, 0)
df_neg4["filename"] = df_neg4["embed_name"].str.split("_").str[0]

In [ ]:
root_folder='../data/audiomoth_Eurasian/embedding/' # Audimoth negative on Eurasian genus (except woodcock) 
df_neg5 = get_embedding_birdnet(root_folder, 0)
df_neg5["filename"] = df_neg5["embed_name"].str.split("_").str[0]

In [ ]:
root_folder='../data/train_data_5/seed1/embedding/' # Baden-Wurttemberg negative records
df_neg6 = get_embedding_birdnet(root_folder, 0)
df_neg6["filename"] = df_neg6["embed_name"].str.split("-").str[0].str[:-1]

In [ ]:
df_neg = pd.concat([df_neg1, df_neg2, df_neg3, df_neg4, df_neg5, df_neg6], ignore_index=True)
#df_neg = pd.concat([df_neg1, df_neg2, df_neg4, df_neg5], ignore_index=True)

In [43]:
len(df_neg)

27914

In [44]:
df_pos['target'] = 1
df_neg['target'] = 0

In [ ]:
from sklearn.model_selection import GroupKFold, cross_val_score

# Combine and shuffle
df_combined = pd.concat([df_pos, df_neg], ignore_index=True)
df_combined = df_combined.sample(frac=1, random_state=22).reset_index(drop=True)

# Features, labels, groups
X = np.vstack(df_combined["embedding"].values)
y = df_combined["target"].values
groups = df_combined["filename"].values

# 5-fold grouped CV
cv = GroupKFold(n_splits=5)

### Cross-Validation

In [ ]:
# Assign fold to every sample
df_combined["fold"] = -1

for fold, (train_idx, val_idx) in enumerate(cv.split(X, y, groups)):
    df_combined.loc[val_idx, "fold"] = fold

SVM

In [31]:
from sklearn.svm import SVC

model = SVC(kernel="rbf", cache_size=300)

accuracies = cross_val_score(
    model,
    X,
    y,
    cv=cv,
    groups=groups,
    scoring="accuracy"
)


ap = cross_val_score(
    model,
    X,
    y,
    cv=cv,
    groups=groups,
    scoring="average_precision"
)

print(f"Accuracy: {accuracies.mean():.4f} ± {accuracies.std(ddof=1):.4f}")
print(f"AP: {ap.mean():.4f} ± {ap.std(ddof=1):.4f}")

Accuracy: 0.9799 ± 0.0031
AP: 0.9746 ± 0.0040


Random Forest

In [40]:
model = RFC(n_jobs = -1)

accuracies = cross_val_score(
    model,
    X,
    y,
    cv=cv,
    groups=groups,
    scoring="accuracy"
)


ap = cross_val_score(
    model,
    X,
    y,
    cv=cv,
    groups=groups,
    scoring="average_precision"
)

print(f"Accuracy: {accuracies.mean():.4f} ± {accuracies.std(ddof=1):.4f}")
print(f"AP: {ap.mean():.4f} ± {ap.std(ddof=1):.4f}")

Accuracy: 0.9418 ± 0.0051
AP: 0.9489 ± 0.0105


XGBoost

In [33]:
model = XGB.XGBClassifier(objective='binary:logistic')

accuracies = cross_val_score(
    model,
    X,
    y,
    cv=cv,
    groups=groups,
    scoring="accuracy"
)


ap = cross_val_score(
    model,
    X,
    y,
    cv=cv,
    groups=groups,
    scoring="average_precision"
)

print(f"Accuracy: {accuracies.mean():.4f} ± {accuracies.std(ddof=1):.4f}")
print(f"AP: {ap.mean():.4f} ± {ap.std(ddof=1):.4f}")

Accuracy: 0.9708 ± 0.0030
AP: 0.9670 ± 0.0045
